In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              REQUIREMENTS — run this cell first              ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

pkgs = [
    "torch-scatter",
    "torch-sparse",
    "torch-geometric",
]

# Install PyG extras that match the Kaggle torch/CUDA version
import torch
torch_ver  = torch.__version__.split("+")[0]          # e.g. "2.1.0"
cuda_ver   = "cu" + torch.version.cuda.replace(".", "") # e.g. "cu118"
pyg_url    = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html"

print(f"torch={torch_ver}  cuda={cuda_ver}")
print(f"Installing PyG wheels from: {pyg_url}\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "torch-scatter", "torch-sparse", "-f", pyg_url],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "torch-geometric"],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "scikit-learn", "pandas", "numpy", "tqdm"],
    check=True
)

print("\n✅ All requirements installed.")
print(f"   GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

torch=2.10.0  cuda=cu128
Installing PyG wheels from: https://data.pyg.org/whl/torch-2.10.0+cu128.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 117.6 MB/s eta 0:00:00

✅ All requirements installed.
   GPUs available: 2
   GPU 0: Tesla T4
   GPU 1: Tesla T4


In [1]:
%ls

facebook_data/  facebook_large.zip


## Dual-graph training (replaces blind merge)

The old notebook **stacked** Facebook and Twitter into one graph (`merge_graphs`), which mixes unrelated nodes and does not match cross-graph alignment papers.

The next cell trains **two separate graphs** with a **shared 3-layer GraphSAGE encoder** (separate linear input projections):

- **Facebook:** `musae_facebook_features.json` binary features + hashed character n-grams from `page_name` in the target CSV.
- **Twitter:** normalized degree, log-degree, sum of neighbor degrees, projected to 128-D.
- **Losses:** node CE on Facebook (`page_type`) and Twitter (degree quartiles), **MSE alignment** between `z_fb[i]` and `z_tw[i]` after matching `|V|`, and **link prediction** (inner product + negative sampling) on each graph.

**Anchors:** SNAP does not ship cross-platform user identities. Index `i` ↔ `i` after `|V_tw| = |V_fb|` is a *weak proxy*. With a CSV of `(fb_node_id, tw_node_id)` pairs, replace the alignment term to use those indices.

Run order: **requirements cell** (PyG), then the code cell below.


In [ ]:
# Dual-graph alignment + link prediction (Facebook + Twitter). Embedded into merged-dataset.ipynb.
# Run from notebook cell, not as __main__.

import os
import json
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, negative_sampling
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("dual_graph")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CFG = dict(
    name_hash_dim=256,
    tw_struct_dim=128,
    hidden=256,
    emb_dim=128,
    num_classes=4,
    dropout=0.2,
    lr=1e-3,
    weight_decay=5e-4,
    epochs=120,
    w_align=2.0,
    w_link_fb=0.5,
    w_link_tw=0.5,
    neg_ratio=1,
    train_frac=0.75,
    facebook_base="/kaggle/working/facebook_data",
    twitter_root="/kaggle/working/twitter_cache",
    twitter_max_nodes=60_000,
)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Device: %s", DEV)


def _find_facebook_files(base):
    found = {}
    for root, _, files in os.walk(base):
        for f in files:
            full = os.path.join(root, f)
            if "edges" in f:
                found["edges"] = full
            if "target" in f:
                found["target"] = full
            if "features" in f:
                found["features"] = full
    if not all(k in found for k in ("edges", "target", "features")):
        raise FileNotFoundError(f"Facebook files incomplete under {base}: {found}")
    return found


def _page_name_trigram_hash(names, dim, ngram=3):
    X = np.zeros((len(names), dim), dtype=np.float32)
    for i, s in enumerate(names):
        t = (s or "").lower()
        if len(t) < ngram:
            h = (hash(t) % dim + dim) % dim
            X[i, h] = 1.0
        else:
            for j in range(len(t) - ngram + 1):
                h = (hash(t[j : j + ngram]) % dim + dim) % dim
                X[i, h] += 1.0
        nrm = X[i].sum()
        if nrm > 0:
            X[i] /= nrm
    return X


def load_facebook_rich(base):
    files = _find_facebook_files(base)
    df_e = pd.read_csv(files["edges"])
    src = torch.tensor(df_e.iloc[:, 0].values, dtype=torch.long)
    dst = torch.tensor(df_e.iloc[:, 1].values, dtype=torch.long)
    ei = to_undirected(torch.stack([src, dst], dim=0))

    with open(files["features"], "r") as f:
        feat_dict = json.load(f)
    nodes = sorted(int(k) for k in feat_dict.keys())
    node_to_idx = {n: i for i, n in enumerate(nodes)}
    max_f = 0
    for feats in feat_dict.values():
        if feats:
            max_f = max(max_f, max(feats))
    Xb = np.zeros((len(nodes), max_f + 1), dtype=np.float32)
    for ks, feats in feat_dict.items():
        ix = node_to_idx[int(ks)]
        for fi in feats or []:
            Xb[ix, fi] = 1.0

    tdf = pd.read_csv(files["target"])
    label_map = {v: i for i, v in enumerate(tdf["page_type"].unique())}
    y = np.zeros(len(nodes), dtype=np.int64)
    names = [""] * len(nodes)
    for _, row in tdf.iterrows():
        nid = int(row["id"])
        if nid not in node_to_idx:
            continue
        ix = node_to_idx[nid]
        y[ix] = label_map[row["page_type"]]
        names[ix] = str(row.get("page_name", "") or "")

    Xname = _page_name_trigram_hash(names, CFG["name_hash_dim"])
    X = np.concatenate([Xb, Xname], axis=1)
    x = torch.from_numpy(X)
    y_t = torch.from_numpy(y)

    remap = torch.full((ei.max().item() + 1,), -1, dtype=torch.long)
    old_ids = torch.tensor(nodes, dtype=torch.long)
    remap[old_ids] = torch.arange(len(nodes), dtype=torch.long)
    e2 = remap[ei]
    ok = (e2 >= 0).all(dim=0)
    ei = e2[:, ok]

    return Data(x=x, edge_index=ei, y=y_t, num_nodes=len(nodes))


def twitter_structural_features(num_nodes, edge_index, dim):
    row, col = edge_index
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, row, torch.ones(row.size(0)))
    deg.scatter_add_(0, col, torch.ones(col.size(0)))
    deg_n = (deg - deg.mean()) / deg.std().clamp_min(1e-6)
    log_d = torch.log(deg + 1.0)
    log_n = (log_d - log_d.mean()) / log_d.std().clamp_min(1e-6)
    neigh_deg = torch.zeros(num_nodes)
    neigh_deg.scatter_add_(0, row, deg[col])
    neigh_deg.scatter_add_(0, col, deg[row])
    nd_n = (neigh_deg - neigh_deg.mean()) / neigh_deg.std().clamp_min(1e-6)
    raw = torch.stack([deg_n, log_n, nd_n, torch.sqrt(deg + 1.0)], dim=1)
    torch.manual_seed(SEED)
    proj = torch.randn(raw.size(1), dim, device=raw.device) * 0.02
    x = raw @ proj
    x = torch.cat([x, torch.sin(deg_n.unsqueeze(1)), torch.cos(deg_n.unsqueeze(1))], dim=1)
    if x.size(1) > dim:
        x = x[:, :dim]
    elif x.size(1) < dim:
        x = F.pad(x, (0, dim - x.size(1)))
    return F.normalize(x, dim=1)


def degree_labels(edge_index, num_nodes, k=4):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    qs = [deg.quantile(q).item() for q in np.linspace(0, 1, k + 1)[1:-1]]
    y = torch.zeros(num_nodes, dtype=torch.long)
    for i, q in enumerate(qs):
        y[deg > q] = i + 1
    return y


def load_twitter_matched(num_target_nodes):
    from torch_geometric.datasets import SNAPDataset

    ds = SNAPDataset(root=CFG["twitter_root"], name="ego-Twitter")
    srcs, dsts = [], []
    cur, offset = 0, 0
    for d in ds:
        if cur >= num_target_nodes:
            break
        if d.edge_index.numel() == 0:
            continue
        n = d.num_nodes
        if cur + n > num_target_nodes:
            n = num_target_nodes - cur
            mask = (d.edge_index[0] < n) & (d.edge_index[1] < n)
            s, t = d.edge_index[:, mask]
        else:
            s, t = d.edge_index
        srcs.append(s + offset)
        dsts.append(t + offset)
        offset += n
        cur += n
        if cur >= CFG["twitter_max_nodes"]:
            break

    ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)], dim=0))
    N = cur
    x = twitter_structural_features(N, ei, CFG["tw_struct_dim"])
    y = degree_labels(ei, N, k=CFG["num_classes"])
    return Data(x=x, edge_index=ei, y=y, num_nodes=N)


class DualGraphNet(nn.Module):
    def __init__(self, in_fb, in_tw, hidden, emb_dim, num_classes, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.proj_fb = nn.Linear(in_fb, hidden)
        self.proj_tw = nn.Linear(in_tw, hidden)
        self.conv1 = SAGEConv(hidden, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, emb_dim)
        self.cls = nn.Linear(emb_dim, num_classes)

    def encode(self, x_proj, edge_index):
        h = F.relu(self.conv1(x_proj, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv3(h, edge_index)
        return h

    def forward(self, x_fb, ei_fb, x_tw, ei_tw):
        z_fb = self.encode(self.proj_fb(x_fb), ei_fb)
        z_tw = self.encode(self.proj_tw(x_tw), ei_tw)
        return z_fb, z_tw, self.cls(z_fb), self.cls(z_tw)


def link_pred_loss(z, pos_edge_index, num_nodes, neg_ratio=1):
    neg = negative_sampling(
        edge_index=pos_edge_index,
        num_nodes=num_nodes,
        num_neg_samples=pos_edge_index.size(1) * neg_ratio,
    )
    s, t = pos_edge_index
    pos_score = (z[s] * z[t]).sum(dim=-1)
    ns, nt = neg
    neg_score = (z[ns] * z[nt]).sum(dim=-1)
    y = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    scores = torch.cat([pos_score, neg_score])
    return F.binary_cross_entropy_with_logits(scores, y)


def run_dual_graph_training():
    fb = load_facebook_rich(CFG["facebook_base"])
    tw = load_twitter_matched(fb.num_nodes)
    assert tw.num_nodes == fb.num_nodes, (tw.num_nodes, fb.num_nodes)
    N = fb.num_nodes
    log.info("Matched graphs: N=%d | d_fb=%d d_tw=%d", N, fb.x.size(1), tw.x.size(1))

    idx = np.arange(N)
    try:
        tr, te = train_test_split(
            idx, train_size=CFG["train_frac"], random_state=SEED, stratify=fb.y.numpy()
        )
    except ValueError:
        tr, te = train_test_split(idx, train_size=CFG["train_frac"], random_state=SEED)
    tr_t = torch.from_numpy(tr).long()
    te_t = torch.from_numpy(te).long()

    model = DualGraphNet(
        fb.x.size(1),
        tw.x.size(1),
        CFG["hidden"],
        CFG["emb_dim"],
        CFG["num_classes"],
        dropout=CFG["dropout"],
    ).to(DEV)

    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    fb_x, fb_ei, fb_y = fb.x.to(DEV), fb.edge_index.to(DEV), fb.y.to(DEV)
    tw_x, tw_ei, tw_y = tw.x.to(DEV), tw.edge_index.to(DEV), tw.y.to(DEV)

    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        opt.zero_grad()
        z_fb, z_tw, lf, lt = model(fb_x, fb_ei, tw_x, tw_ei)

        loss_node_fb = F.cross_entropy(lf[tr_t], fb_y[tr_t])
        loss_node_tw = F.cross_entropy(lt[tr_t], tw_y[tr_t])
        loss_align = F.mse_loss(z_fb, z_tw)
        lp_fb = link_pred_loss(z_fb, fb_ei, N, CFG["neg_ratio"])
        lp_tw = link_pred_loss(z_tw, tw_ei, N, CFG["neg_ratio"])

        loss = (
            loss_node_fb
            + loss_node_tw
            + CFG["w_align"] * loss_align
            + CFG["w_link_fb"] * lp_fb
            + CFG["w_link_tw"] * lp_tw
        )
        loss.backward()
        opt.step()

        if epoch % 10 == 0 or epoch == 1:
            log.info(
                "Ep %d | total %.4f | CE_fb %.4f CE_tw %.4f | align %.4f | lp %.4f",
                epoch,
                loss.item(),
                loss_node_fb.item(),
                loss_node_tw.item(),
                loss_align.item(),
                lp_fb.item() + lp_tw.item(),
            )

    model.eval()
    with torch.no_grad():
        z_fb, z_tw, lf, lt = model(fb_x, fb_ei, tw_x, tw_ei)

    def report(logits, y_true, name):
        proba = torch.softmax(logits, dim=1).cpu().numpy()
        pred = logits.argmax(dim=1).cpu().numpy()
        yt = y_true.cpu().numpy()
        acc = accuracy_score(yt, pred)
        f1 = f1_score(yt, pred, average="macro", zero_division=0)
        c = proba.shape[1]
        try:
            auc = (
                roc_auc_score(yt, proba[:, 1])
                if c == 2
                else roc_auc_score(yt, proba, multi_class="ovr", average="macro", labels=np.arange(c))
            )
        except ValueError:
            auc = float("nan")
        print(f"{name}  Acc: {acc:.4f}  F1(macro): {f1:.4f}  AUC(OVR): {auc:.4f}")

    print("\n=== Held-out node classification (test idx) ===")
    report(lf[te_t], fb_y[te_t], "Facebook (page_type)")
    report(lt[te_t], tw_y[te_t], "Twitter (degree tiers)")

run_dual_graph_training()


In [1]:
import shutil

shutil.make_archive('/kaggle/working_output', 'zip', '/kaggle/working')

'/kaggle/working_output.zip'

In [2]:
from IPython.display import FileLink
FileLink('/kaggle/working_output.zip')

/kaggle/working_output.zip

In [3]:
import os, zipfile

zip_path = '/kaggle/working_clean.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('/kaggle/working'):
        for f in files:
            if f.endswith('.pt') or f.endswith('.npy'):  # skip large files if needed
                continue
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, '/kaggle/working'))

print("Created:", zip_path)

Created: /kaggle/working_clean.zip


In [4]:
from IPython.display import FileLink
FileLink('/kaggle/working_clean.zip')

/kaggle/working_clean.zip

In [5]:
!mv /kaggle/working_clean.zip /kaggle/working/